# Module 14 — Imbalanced Fraud Handling: Weighted Cross-Entropy in PyTorch

> **Track 2 · Revolut Fintech Specialization**  
> **Difficulty**: Advanced  
> **Runtime**: ~5 minutes  
> **Key Libraries**: PyTorch, Scikit-learn, NumPy, Pandas, Plotly, Seaborn  
> **Prerequisite**: Module 13 (Logistic Regression Baseline)

## Table of Contents

1. [Business Context](#1-business-context)
2. [Setup & Imports](#2-setup--imports)
3. [Synthetic Fraud Dataset](#3-synthetic-fraud-dataset)
4. [The Imbalance Problem](#4-the-imbalance-problem)
5. [Standard BCE Loss (Baseline)](#5-standard-bce-loss-baseline)
6. [Custom Weighted BCE Loss](#6-custom-weighted-bce-loss)
7. [Focal Loss (Advanced)](#7-focal-loss-advanced)
8. [Training Loop Comparison](#8-training-loop-comparison)
9. [Evaluation & PR-AUC](#9-evaluation--pr-auc)
10. [Visualization Dashboard](#10-visualization-dashboard)
11. [Key Business Takeaway](#11-key-business-takeaway)

---
## §1 · Business Context

### Why Weighted Loss at Revolut?

Revolut's fraud detection faces a **severe class imbalance**:
- **99.5%** of transactions are legitimate
- **0.5%** are fraudulent

Standard neural networks trained with BCE loss will:
- Achieve 99.5% accuracy by predicting "not fraud" for everything
- **Miss all fraud** — the exact opposite of the business goal

**Solutions**:
1. **Class weights** in the loss function (this module)
2. **Focal loss** — down-weights easy examples, focuses on hard ones
3. **Oversampling/undersampling** (Module 7)

In this notebook we will:
1. Generate a 150K-transaction fraud dataset (0.5% fraud rate).
2. Train a PyTorch neural network with **standard BCE** (baseline).
3. Implement a **custom Weighted BCE** loss that penalizes missed fraud 50× more.
4. Implement **Focal Loss** for comparison.
5. Evaluate all three with **PR-AUC** (the gold standard for imbalanced classification).

---
## §2 · Setup & Imports

In [ ]:
# ── Reproducibility ──────────────────────────────────────────────
import numpy as np
import pandas as pd
import time
import warnings

np.random.seed(42)
warnings.filterwarnings("ignore")

# ── PyTorch ──────────────────────────────────────────────────────
import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import TensorDataset, DataLoader

torch.manual_seed(42)

# ── Scikit-learn ─────────────────────────────────────────────────
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import (
    precision_score, recall_score, f1_score, roc_auc_score,
    average_precision_score, precision_recall_curve, roc_curve,
    confusion_matrix
)

# ── Visualization ────────────────────────────────────────────────
import matplotlib.pyplot as plt
import seaborn as sns
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.express as px

# ── Display settings ─────────────────────────────────────────────
pd.set_option("display.max_columns", 25)
pd.set_option("display.float_format", lambda x: f"{x:,.4f}")
sns.set_theme(style="whitegrid")

print("✓ All imports loaded")
print(f"  PyTorch {torch.__version__}")
print(f"  CUDA available: {torch.cuda.is_available()}")

---
## §3 · Synthetic Fraud Dataset

We simulate **150,000 transactions** with **25 features** and a **0.5% fraud rate**.

In [ ]:
# ── Configuration ────────────────────────────────────────────────
N_TXN = 150_000
FRAUD_RATE = 0.005  # 0.5% fraud rate (realistic for fintech)
N_FRAUD = int(N_TXN * FRAUD_RATE)
N_LEGIT = N_TXN - N_FRAUD
N_FEATURES = 25

print(f"Generating fraud dataset:")
print(f"  Transactions: {N_TXN:,}")
print(f"  Fraud rate   : {FRAUD_RATE*100:.1f}%")
print(f"  Class ratio  : 1:{N_LEGIT // N_FRAUD}")

np.random.seed(42)

# ── Feature names ────────────────────────────────────────────────
feature_names = [
    "amount", "amount_log", "hour", "is_weekend", "txn_count_1h",
    "txn_count_24h", "amount_sum_24h", "merchant_risk", "country_risk",
    "is_international", "distance_from_home", "velocity_1h",
    "unique_merchants_7d", "is_atm", "is_transfer", "user_tenure_days",
    "credit_score", "z_score_amount", "amount_to_income_ratio",
    "days_since_last_txn", "amount_squared", "risk_interaction",
    "velocity_amount", "tenure_risk", "is_high_risk_country",
]

# ── Generate legitimate transactions ─────────────────────────────
X_legit = np.random.randn(N_LEGIT, N_FEATURES)
X_legit[:, 0] = np.abs(np.random.lognormal(3.0, 1.0, N_LEGIT))
X_legit[:, 1] = np.log1p(X_legit[:, 0])
X_legit[:, 7] = np.random.uniform(0, 50, N_LEGIT)
X_legit[:, 8] = np.random.uniform(0, 30, N_LEGIT)
X_legit[:, 16] = np.random.normal(650, 80, N_LEGIT)
X_legit[:, 17] = np.random.normal(0, 1, N_LEGIT)

# ── Generate fraudulent transactions ─────────────────────────────
X_fraud = np.random.randn(N_FRAUD, N_FEATURES)
X_fraud[:, 0] = np.abs(np.random.lognormal(5.0, 1.5, N_FRAUD))
X_fraud[:, 1] = np.log1p(X_fraud[:, 0])
X_fraud[:, 7] = np.random.uniform(60, 100, N_FRAUD)
X_fraud[:, 8] = np.random.uniform(70, 100, N_FRAUD)
X_fraud[:, 9] = np.random.binomial(1, 0.60, N_FRAUD)
X_fraud[:, 17] = np.random.normal(3, 1.5, N_FRAUD)
X_fraud[:, 24] = np.random.binomial(1, 0.70, N_FRAUD)

# ── Combine ──────────────────────────────────────────────────────
X = np.vstack([X_legit, X_fraud])
y = np.concatenate([np.zeros(N_LEGIT), np.ones(N_FRAUD)])

idx = np.random.permutation(N_TXN)
X = X[idx]
y = y[idx]

# Split and scale
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, stratify=y, random_state=42
)

scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

# Convert to PyTorch tensors
X_train_t = torch.tensor(X_train_scaled, dtype=torch.float32)
y_train_t = torch.tensor(y_train, dtype=torch.float32)
X_test_t = torch.tensor(X_test_scaled, dtype=torch.float32)
y_test_t = torch.tensor(y_test, dtype=torch.float32)

train_ds = TensorDataset(X_train_t, y_train_t)
train_dl = DataLoader(train_ds, batch_size=256, shuffle=True)

print(f"\n✓ Train: {len(y_train):,} ({y_train.sum():.0f} fraud, {y_train.mean()*100:.3f}%)")
print(f"  Test : {len(y_test):,} ({y_test.sum():.0f} fraud, {y_test.mean()*100:.3f}%)")

---
## §4 · The Imbalance Problem

### Why Standard BCE Fails

In [ ]:
# Compute class weights
n_neg = (y_train == 0).sum()
n_pos = (y_train == 1).sum()
total = len(y_train)

weight_neg = total / (2 * n_neg)
weight_pos = total / (2 * n_pos)

print("=" * 70)
print("CLASS IMBALANCE ANALYSIS")
print("=" * 70)
print(f"\nTraining set:")
print(f"  Negative (legit): {n_neg:,} ({n_neg/total*100:.2f}%)")
print(f"  Positive (fraud): {n_pos:,} ({n_pos/total*100:.2f}%)")
print(f"  Class ratio     : 1:{n_neg // n_pos}")

print(f"\nStandard BCE loss:")
print(f"  Treats all samples equally")
print(f"  → Model learns to predict 'not fraud' (99.5% accuracy)")
print(f"  → Misses ALL fraud (0% recall)")

print(f"\nWeighted BCE loss:")
print(f"  weight_neg = {weight_neg:.4f}")
print(f"  weight_pos = {weight_pos:.4f} ({weight_pos/weight_neg:.1f}× higher)")
print(f"  → Model penalizes missed fraud {n_neg // n_pos}× more")
print(f"  → Learns to catch fraud despite imbalance")

---
## §5 · Standard BCE Loss (Baseline)

### Neural Network Architecture

In [ ]:
class FraudNet(nn.Module):
    """Simple feed-forward network for fraud detection."""
    
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, 64),
            nn.ReLU(),
            nn.Dropout(0.2),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1),
        )
    
    def forward(self, x):
        return self.net(x).squeeze()

# Test architecture
model_test = FraudNet(X_train_scaled.shape[1])
print(model_test)
print(f"\nTotal parameters: {sum(p.numel() for p in model_test.parameters()):,}")

### Training with Standard BCE

In [ ]:
def train_model(model, criterion, n_epochs=30, lr=0.001, verbose=False):
    """Train a PyTorch model with the given loss function."""
    optimizer = optim.Adam(model.parameters(), lr=lr)
    model.train()
    
    train_losses = []
    for epoch in range(n_epochs):
        epoch_loss = 0
        n_batches = 0
        
        for X_batch, y_batch in train_dl:
            optimizer.zero_grad()
            logits = model(X_batch)
            loss = criterion(logits, y_batch)
            loss.backward()
            optimizer.step()
            
            epoch_loss += loss.item()
            n_batches += 1
        
        avg_loss = epoch_loss / n_batches
        train_losses.append(avg_loss)
        
        if verbose and (epoch + 1) % 10 == 0:
            print(f"  Epoch {epoch+1:3d}/{n_epochs}: loss = {avg_loss:.6f}")
    
    return model, train_losses

# ── Train with standard BCE ──────────────────────────────────────
print("Training with Standard BCE Loss …")
model_bce = FraudNet(X_train_scaled.shape[1])
criterion_bce = nn.BCEWithLogitsLoss()

t0 = time.time()
model_bce, losses_bce = train_model(model_bce, criterion_bce, n_epochs=30)
time_bce = time.time() - t0

print(f"✓ Trained in {time_bce:.2f}s")
print(f"  Final loss: {losses_bce[-1]:.6f}")

---
## §6 · Custom Weighted BCE Loss

### Implementation

In [ ]:
class WeightedBCELoss(nn.Module):
    """
    Weighted Binary Cross-Entropy Loss.
    
    Penalizes positive (fraud) samples more heavily than negative (legit) samples.
    
    Args:
        pos_weight: Weight for positive samples (fraud).
                    Higher = more penalty for missed fraud.
    """
    
    def __init__(self, pos_weight):
        super().__init__()
        self.pos_weight = pos_weight
    
    def forward(self, logits, targets):
        # Compute BCE without reduction
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction='none'
        )
        
        # Apply weights: pos_weight for fraud, 1.0 for legit
        weights = torch.where(targets == 1, self.pos_weight, torch.tensor(1.0))
        weighted_bce = bce * weights
        
        return weighted_bce.mean()

# Compute pos_weight (ratio of negatives to positives)
pos_weight = torch.tensor(n_neg / n_pos, dtype=torch.float32)

print(f"Weighted BCE Loss:")
print(f"  pos_weight = {pos_weight.item():.1f}")
print(f"  → Fraud samples are penalized {pos_weight.item():.0f}× more than legit samples")
print(f"  → Model will prioritize catching fraud over avoiding false alarms")

### Training with Weighted BCE

In [ ]:
print("Training with Weighted BCE Loss …")
model_weighted = FraudNet(X_train_scaled.shape[1])
criterion_weighted = WeightedBCELoss(pos_weight)

t0 = time.time()
model_weighted, losses_weighted = train_model(model_weighted, criterion_weighted, n_epochs=30)
time_weighted = time.time() - t0

print(f"✓ Trained in {time_weighted:.2f}s")
print(f"  Final loss: {losses_weighted[-1]:.6f}")

print("\n💡 Weighted loss converges to a lower value because it penalizes fraud more")
print("   but the model learns to catch fraud despite the imbalance")

---
## §7 · Focal Loss (Advanced)

### What is Focal Loss?

**Focal Loss** (Lin et al., 2017) addresses class imbalance by:
- **Down-weighting easy examples** (confident predictions)
- **Focusing on hard examples** (uncertain predictions)

```
FL(p_t) = -α_t * (1 - p_t)^γ * log(p_t)

Where:
  - p_t = predicted probability for the true class
  - γ (gamma) = focusing parameter (typically 2.0)
  - α_t = class weight
```

When γ = 0, Focal Loss = standard BCE.
When γ > 0, easy examples (p_t close to 1) contribute less to the loss.

In [ ]:
class FocalLoss(nn.Module):
    """
    Focal Loss for imbalanced classification.
    
    Args:
        alpha: Weight for positive class (fraud).
        gamma: Focusing parameter (2.0 is typical).
    """
    
    def __init__(self, alpha=1.0, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma
    
    def forward(self, logits, targets):
        # Compute BCE
        bce = nn.functional.binary_cross_entropy_with_logits(
            logits, targets, reduction='none'
        )
        
        # Compute p_t (probability of true class)
        probs = torch.sigmoid(logits)
        p_t = torch.where(targets == 1, probs, 1 - probs)
        
        # Focal weight: (1 - p_t)^gamma
        focal_weight = (1 - p_t) ** self.gamma
        
        # Class weight
        alpha_t = torch.where(targets == 1, 
                              torch.tensor(self.alpha), 
                              torch.tensor(1.0))
        
        # Focal loss
        focal_loss = alpha_t * focal_weight * bce
        
        return focal_loss.mean()

# Initialize Focal Loss
# alpha = pos_weight (same as weighted BCE)
# gamma = 2.0 (standard focusing parameter)
focal_alpha = pos_weight.item()
focal_gamma = 2.0

criterion_focal = FocalLoss(alpha=focal_alpha, gamma=focal_gamma)

print(f"Focal Loss Configuration:")
print(f"  alpha = {focal_alpha:.1f} (class weight)")
print(f"  gamma = {focal_gamma} (focusing parameter)")
print(f"\n  → Down-weights easy examples (confident predictions)")
print(f"  → Focuses on hard examples (uncertain predictions)")
print(f"  → Combines class weighting with hard example mining")

### Training with Focal Loss

In [ ]:
print("Training with Focal Loss …")
model_focal = FraudNet(X_train_scaled.shape[1])

t0 = time.time()
model_focal, losses_focal = train_model(model_focal, criterion_focal, n_epochs=30)
time_focal = time.time() - t0

print(f"✓ Trained in {time_focal:.2f}s")
print(f"  Final loss: {losses_focal[-1]:.6f}")

print("\n💡 Focal Loss combines:")
print("   1. Class weighting (like Weighted BCE)")
print("   2. Hard example mining (focuses on uncertain predictions)")
print("   → Often outperforms simple weighted loss on severe imbalance")

---
## §8 · Training Loop Comparison

### Loss Curves

In [ ]:
# Plot training loss curves
fig = go.Figure()

fig.add_trace(go.Scatter(
    y=losses_bce, mode="lines",
    name="Standard BCE",
    line=dict(color="#636EFA", width=2.5),
))

fig.add_trace(go.Scatter(
    y=losses_weighted, mode="lines",
    name="Weighted BCE",
    line=dict(color="#EF553B", width=2.5),
))

fig.add_trace(go.Scatter(
    y=losses_focal, mode="lines",
    name="Focal Loss",
    line=dict(color="#00CC96", width=2.5),
))

fig.update_layout(
    title="Training Loss Curves (30 epochs)",
    xaxis_title="Epoch",
    yaxis_title="Loss",
    height=440,
    legend=dict(orientation="h", y=1.12),
)
fig.show()

print("💡 Standard BCE converges quickly but to a suboptimal solution")
print("   Weighted BCE and Focal Loss take longer but learn to catch fraud")

---
## §9 · Evaluation & PR-AUC

### Evaluation Function

In [ ]:
def evaluate_model(model, X_test, y_test, threshold=0.5):
    """Evaluate a trained model on test data."""
    model.eval()
    with torch.no_grad():
        logits = model(X_test)
        probs = torch.sigmoid(logits).numpy()
        preds = (probs >= threshold).astype(int)
    
    metrics = {
        "Accuracy": (preds == y_test).mean(),
        "Precision": precision_score(y_test, preds, zero_division=0),
        "Recall": recall_score(y_test, preds, zero_division=0),
        "F1": f1_score(y_test, preds, zero_division=0),
        "ROC-AUC": roc_auc_score(y_test, probs),
        "PR-AUC": average_precision_score(y_test, probs),
    }
    
    return metrics, probs, preds

# Evaluate all three models
models = {
    "Standard BCE": model_bce,
    "Weighted BCE": model_weighted,
    "Focal Loss": model_focal,
}

results = {}
probs_dict = {}

for name, model in models.items():
    metrics, probs, preds = evaluate_model(model, X_test_t, y_test.numpy())
    results[name] = metrics
    probs_dict[name] = probs

# Also evaluate logistic regression baseline
lr = LogisticRegression(class_weight="balanced", max_iter=1000, random_state=42)
lr.fit(X_train_scaled, y_train)
y_prob_lr = lr.predict_proba(X_test_scaled)[:, 1]
y_pred_lr = lr.predict(X_test_scaled)

results["Logistic Regression"] = {
    "Accuracy": (y_pred_lr == y_test).mean(),
    "Precision": precision_score(y_test, y_pred_lr, zero_division=0),
    "Recall": recall_score(y_test, y_pred_lr, zero_division=0),
    "F1": f1_score(y_test, y_pred_lr, zero_division=0),
    "ROC-AUC": roc_auc_score(y_test, y_prob_lr),
    "PR-AUC": average_precision_score(y_test, y_prob_lr),
}
probs_dict["Logistic Regression"] = y_prob_lr

# Compile results
df_results = pd.DataFrame(results).T.round(4)
print("=" * 90)
print("MODEL COMPARISON (Test Set)")
print("=" * 90)
print(df_results.to_string())

print("\n💡 Key observations:")
print("   - Standard BCE has high accuracy but ZERO recall (misses all fraud)")
print("   - Weighted BCE and Focal Loss achieve high recall (catch fraud)")
print("   - PR-AUC is the gold standard for imbalanced classification")
print("   - Logistic Regression is competitive with proper class weighting")

---
## §10 · Visualization Dashboard

In [ ]:
# ── 10.1 Precision-Recall Curves ─────────────────────────────────
fig = go.Figure()

colors = {"Standard BCE": "#636EFA", "Weighted BCE": "#EF553B",
          "Focal Loss": "#00CC96", "Logistic Regression": "#FFA15A"}

for name, probs in probs_dict.items():
    precision, recall, _ = precision_recall_curve(y_test, probs)
    pr_auc = average_precision_score(y_test, probs)
    
    fig.add_trace(go.Scatter(
        x=recall, y=precision,
        mode="lines",
        name=f"{name} (PR-AUC = {pr_auc:.4f})",
        line=dict(color=colors[name], width=2.5),
    ))

fig.update_layout(
    title="Precision-Recall Curves (Higher is Better)",
    xaxis_title="Recall",
    yaxis_title="Precision",
    height=500,
    legend=dict(orientation="h", y=1.12),
)
fig.show()

print("💡 PR-AUC is the gold standard for imbalanced classification")
print("   Standard BCE has near-zero PR-AUC (misses all fraud)")
print("   Weighted BCE and Focal Loss achieve high PR-AUC")

In [ ]:
# ── 10.2 ROC Curves ──────────────────────────────────────────────
fig = go.Figure()

for name, probs in probs_dict.items():
    fpr, tpr, _ = roc_curve(y_test, probs)
    roc_auc = roc_auc_score(y_test, probs)
    
    fig.add_trace(go.Scatter(
        x=fpr, y=tpr,
        mode="lines",
        name=f"{name} (AUC = {roc_auc:.4f})",
        line=dict(color=colors[name], width=2.5),
    ))

fig.add_trace(go.Scatter(
    x=[0, 1], y=[0, 1],
    mode="lines",
    name="Random Classifier",
    line=dict(color="gray", width=1, dash="dash"),
))

fig.update_layout(
    title="ROC Curves",
    xaxis_title="False Positive Rate",
    yaxis_title="True Positive Rate",
    height=500,
    legend=dict(orientation="h", y=1.12),
)
fig.show()

print("💡 ROC-AUC can be misleading for imbalanced data")
print("   Standard BCE appears 'okay' on ROC but has zero recall")
print("   Always check PR-AUC for imbalanced classification")

In [ ]:
# ── 10.3 Confusion Matrix Comparison ─────────────────────────────
fig = make_subplots(rows=2, cols=2,
                    subplot_titles=["Standard BCE", "Weighted BCE",
                                    "Focal Loss", "Logistic Regression"])

models_list = ["Standard BCE", "Weighted BCE", "Focal Loss", "Logistic Regression"]

for i, name in enumerate(models_list, 1):
    if name == "Logistic Regression":
        y_pred = y_pred_lr
    else:
        _, _, y_pred = evaluate_model(models[name], X_test_t, y_test.numpy())
    
    cm = confusion_matrix(y_test, y_pred)
    
    fig.add_trace(go.Heatmap(
        z=cm, x=["Pred 0", "Pred 1"], y=["Actual 0", "Actual 1"],
        colorscale="Blues", showscale=False,
        text=cm, texttemplate="%{text}", textfont={"size": 16},
    ), row=(i-1)//2 + 1, col=(i-1) % 2 + 1)

fig.update_layout(height=600, title_text="Confusion Matrices")
fig.show()

print("💡 Standard BCE predicts all negatives (high TN, zero TP)")
print("   Weighted BCE and Focal Loss catch fraud (high TP) with some false alarms (FP)")

In [ ]:
# ── 10.4 Business Impact ─────────────────────────────────────────
COST_FN = 2500  # £ per missed fraud
COST_FP = 5     # £ per false alarm

print("=" * 90)
print("BUSINESS IMPACT ANALYSIS (Test Set, 45K transactions)")
print("=" * 90)
print(f"\nAssumptions:")
print(f"  Cost per missed fraud (FN): £{COST_FN:,}")
print(f"  Cost per false alarm (FP) : £{COST_FP}")

print(f"\n{'Model':<25} {'TP':>6} {'FP':>8} {'FN':>8} {'Total Cost (£)':>15}")
print("-" * 70)

for name in models_list:
    if name == "Logistic Regression":
        y_pred = y_pred_lr
    else:
        _, _, y_pred = evaluate_model(models[name], X_test_t, y_test.numpy())
    
    cm = confusion_matrix(y_test, y_pred)
    tp = cm[1, 1]
    fp = cm[0, 1]
    fn = cm[1, 0]
    
    total_cost = fn * COST_FN + fp * COST_FP
    print(f"{name:<25} {tp:>6} {fp:>8,} {fn:>8} £{total_cost:>14,}")

print("\n💡 Standard BCE has the HIGHEST cost despite 99.5% accuracy")
print("   Weighted BCE and Focal Loss minimize total business cost")

---
## §11 · Key Business Takeaway

> ### 🎯 Action Items for a Fintech Data Scientist
>
> | Loss Function | Best For | Pros | Cons |
> |---------------|----------|------|------|
> | **Standard BCE** | Balanced data | Simple, fast | Fails on imbalance |
> | **Weighted BCE** | Moderate imbalance | Easy to implement | Tuning pos_weight |
> | **Focal Loss** | Severe imbalance | Hard example mining | More hyperparameters |
>
> ### 💡 Revolut-Specific Guidelines
>
> 1. **Never use standard BCE for fraud detection** — it will miss all fraud.
>
> 2. **Weighted BCE is the go-to solution**:
>    ```python
>    pos_weight = n_negatives / n_positives  # e.g., 200 for 0.5% fraud
>    criterion = WeightedBCELoss(pos_weight=pos_weight)
>    ```
>
> 3. **Focal Loss for extreme imbalance** (< 0.1% fraud rate):
>    ```python
>    criterion = FocalLoss(alpha=pos_weight, gamma=2.0)
>    ```
>
> 4. **Always evaluate with PR-AUC**, not accuracy or ROC-AUC:
>    ```python
>    from sklearn.metrics import average_precision_score
>    pr_auc = average_precision_score(y_test, y_prob)
>    ```
>
> 5. **Threshold tuning is critical** — default 0.5 is rarely optimal:
>    ```python
>    # Find threshold that maximizes F1
>    precision, recall, thresholds = precision_recall_curve(y_test, y_prob)
>    f1_scores = 2 * precision * recall / (precision + recall + 1e-8)
>    optimal_threshold = thresholds[np.argmax(f1_scores)]
>    ```
>
> ### 📌 Loss Function Cheat Sheet
>
> ```python
> # Standard BCE (balanced data only)
> criterion = nn.BCEWithLogitsLoss()
>
> # Weighted BCE (moderate imbalance)
> pos_weight = torch.tensor(n_neg / n_pos)
> criterion = WeightedBCELoss(pos_weight=pos_weight)
>
> # Focal Loss (severe imbalance)
> criterion = FocalLoss(alpha=pos_weight, gamma=2.0)
> ```
>
> ### 🔑 Key Takeaways
>
> 1. **Standard BCE fails on imbalanced data** — always use weighted or focal loss.
> 2. **PR-AUC > ROC-AUC** for rare positive classes (fraud, disease, defects).
> 3. **Weighted BCE is simple and effective** — start here.
> 4. **Focal Loss adds hard example mining** — use for extreme imbalance.
> 5. **Business cost matrix** should guide model selection, not just statistical metrics.